In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ConvINAct(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size=k, padding=p),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.LeakyReLU(0.1, inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class ResidualBlock3D(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv1 = ConvINAct(ch, ch, k=3, p=1)
        self.conv2 = nn.Sequential(
            nn.Conv3d(ch, ch, kernel_size=3, padding=1),
            nn.InstanceNorm3d(ch, affine=True),
        )
        self.act = nn.LeakyReLU(0.1, inplace=True)

    def forward(self, x):
        out = self.conv1(x)
        out = self.conv2(out)
        return self.act(x + out)


class EarlyMixWideResidualNet(nn.Module):
    """
    Early global channel mixing model.

    Input:
        x        [B,1536,16,8,8]
        baseline [B,2,16,8,8]

    Output:
        pred     [B,2,16,8,8]

    Strategy:
        1536 RF channels are globally mixed at the first 1x1x1 layer.
        Then small 3D residual blocks learn spatial correction.
    """

    def __init__(self, in_channels=1536, stem_channels=128, mid_channels=64, out_channels=2):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv3d(in_channels, stem_channels, kernel_size=1, padding=0),
            nn.InstanceNorm3d(stem_channels, affine=True),
            nn.LeakyReLU(0.1, inplace=True),
        )

        self.res1 = ResidualBlock3D(stem_channels)
        self.res2 = ResidualBlock3D(stem_channels)

        self.down = nn.Sequential(
            nn.Conv3d(stem_channels, mid_channels, kernel_size=1, padding=0),
            nn.InstanceNorm3d(mid_channels, affine=True),
            nn.LeakyReLU(0.1, inplace=True),
        )

        self.res3 = ResidualBlock3D(mid_channels)

        self.out = nn.Conv3d(mid_channels, out_channels, kernel_size=1, padding=0)

        # Start from baseline.
        nn.init.zeros_(self.out.weight)
        nn.init.zeros_(self.out.bias)

    def forward(self, x, baseline):
        feat = self.stem(x)
        feat = self.res1(feat)
        feat = self.res2(feat)
        feat = self.down(feat)
        feat = self.res3(feat)

        residual = self.out(feat)
        pred = baseline + residual
        return pred

In [2]:
from pathlib import Path
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np


def complex_abs_2ch(x):
    """
    x: [B, 2, Z, X, Y]
    """
    return torch.sqrt(x[:, 0:1] ** 2 + x[:, 1:2] ** 2 + 1e-8)


def infer_category_from_path(path):
    p = str(path).lower()
    for cat in ["carotid", "muscle", "phantom", "simu_point"]:
        if cat in p:
            return cat
    return "unknown"


@torch.no_grad()
def evaluate_full_test_set(
    model,
    dataset,
    device,
    batch_size=4,
    save_csv_path=None,
):
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )

    model.eval()

    rows = []

    for batch in loader:
        x = batch["input"].to(device, non_blocking=True)
        y = batch["label"].to(device, non_blocking=True)
        b = batch["baseline"].to(device, non_blocking=True)

        scale = batch["scale"].to(device)
        if scale.ndim == 1:
            scale = scale.view(-1, 1, 1, 1, 1)

        pred = model(x, b)

        # Denormalize if dataset used normalize=True.
        pred_raw = pred * scale
        y_raw = y * scale
        b_raw = b * scale

        # Per-sample complex L1.
        pred_l1 = torch.mean(torch.abs(pred_raw - y_raw), dim=(1, 2, 3, 4))
        base_l1 = torch.mean(torch.abs(b_raw - y_raw), dim=(1, 2, 3, 4))

        # Per-sample envelope L1.
        pred_abs = complex_abs_2ch(pred_raw)
        y_abs = complex_abs_2ch(y_raw)
        b_abs = complex_abs_2ch(b_raw)

        pred_abs_l1 = torch.mean(torch.abs(pred_abs - y_abs), dim=(1, 2, 3, 4))
        base_abs_l1 = torch.mean(torch.abs(b_abs - y_abs), dim=(1, 2, 3, 4))

        paths = batch["path"]

        for i, path in enumerate(paths):
            pl1 = float(pred_l1[i].cpu())
            bl1 = float(base_l1[i].cpu())
            pa1 = float(pred_abs_l1[i].cpu())
            ba1 = float(base_abs_l1[i].cpu())

            rows.append({
                "path": path,
                "category": infer_category_from_path(path),

                "pred_complex_l1": pl1,
                "base_complex_l1": bl1,
                "complex_improvement": 1.0 - pl1 / (bl1 + 1e-12),
                "complex_better": pl1 < bl1,

                "pred_abs_l1": pa1,
                "base_abs_l1": ba1,
                "abs_improvement": 1.0 - pa1 / (ba1 + 1e-12),
                "abs_better": pa1 < ba1,
            })

    df = pd.DataFrame(rows)

    if save_csv_path is not None:
        save_csv_path = Path(save_csv_path)
        save_csv_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(save_csv_path, index=False)
        print(f"Saved per-sample metrics to: {save_csv_path}")

    return df


def summarize_test_metrics(df):
    overall = {
        "n": len(df),

        "complex_pred_mean": df["pred_complex_l1"].mean(),
        "complex_base_mean": df["base_complex_l1"].mean(),
        "complex_improvement_mean": df["complex_improvement"].mean(),
        "complex_better_rate": df["complex_better"].mean(),

        "abs_pred_mean": df["pred_abs_l1"].mean(),
        "abs_base_mean": df["base_abs_l1"].mean(),
        "abs_improvement_mean": df["abs_improvement"].mean(),
        "abs_better_rate": df["abs_better"].mean(),
    }

    print("\n================ Overall test summary ================")
    print(f"Samples: {overall['n']}")

    print("\n[Complex L1]")
    print(f"pred mean       : {overall['complex_pred_mean']:.4e}")
    print(f"baseline mean   : {overall['complex_base_mean']:.4e}")
    print(f"mean improvement: {overall['complex_improvement_mean']*100:.2f}%")
    print(f"better rate     : {overall['complex_better_rate']*100:.2f}%")

    print("\n[Envelope abs L1]")
    print(f"pred mean       : {overall['abs_pred_mean']:.4e}")
    print(f"baseline mean   : {overall['abs_base_mean']:.4e}")
    print(f"mean improvement: {overall['abs_improvement_mean']*100:.2f}%")
    print(f"better rate     : {overall['abs_better_rate']*100:.2f}%")

    print("\n================ Per-category summary ================")
    cat_summary = df.groupby("category").agg(
        n=("path", "count"),

        complex_pred_mean=("pred_complex_l1", "mean"),
        complex_base_mean=("base_complex_l1", "mean"),
        complex_improvement_mean=("complex_improvement", "mean"),
        complex_better_rate=("complex_better", "mean"),

        abs_pred_mean=("pred_abs_l1", "mean"),
        abs_base_mean=("base_abs_l1", "mean"),
        abs_improvement_mean=("abs_improvement", "mean"),
        abs_better_rate=("abs_better", "mean"),
    )

    display(cat_summary)

    return overall, cat_summary

In [3]:
from pathlib import Path
from torch.utils.data import DataLoader

from rf_learning_dataset import RFLearningDataset


root_dir = Path("/home/liujia/3DSSIM_1/RF_Image/Data")

train_root = root_dir / "train"
val_root   = root_dir / "val"
test_root  = root_dir / "test"

train_set = RFLearningDataset(
    root_dir=train_root,
    sample_group="/sample_000001",
    normalize=True,
)

val_set = RFLearningDataset(
    root_dir=val_root,
    sample_group="/sample_000001",
    normalize=True,
)

test_set = RFLearningDataset(
    root_dir=test_root,
    sample_group="/sample_000001",
    normalize=True,
)

train_loader = DataLoader(
    train_set,
    batch_size=4,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)

val_loader = DataLoader(
    val_set,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

test_loader = DataLoader(
    test_set,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

print("Train samples:", len(train_set))
print("Val samples  :", len(val_set))
print("Test samples :", len(test_set))

batch = next(iter(train_loader))
print("input   :", batch["input"].shape)
print("label   :", batch["label"].shape)
print("baseline:", batch["baseline"].shape)

Train samples: 200
Val samples  : 40
Test samples : 40
input   : torch.Size([4, 1536, 16, 8, 8])
label   : torch.Size([4, 2, 16, 8, 8])
baseline: torch.Size([4, 2, 16, 8, 8])


In [4]:
import random
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn.functional as F


def seed_everything(seed=20260522):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def complex_abs_2ch(x):
    return torch.sqrt(x[:, 0:1] ** 2 + x[:, 1:2] ** 2 + 1e-8)


def compute_loss(pred, label):
    loss_l1 = F.l1_loss(pred, label)

    pred_abs = complex_abs_2ch(pred)
    label_abs = complex_abs_2ch(label)
    loss_abs = F.l1_loss(pred_abs, label_abs)

    loss = loss_l1 + 0.1 * loss_abs
    return loss, loss_l1, loss_abs


@torch.no_grad()
def evaluate_normalized(model, loader, device):
    model.eval()

    total_loss = 0.0
    total_l1 = 0.0
    total_abs = 0.0
    total_base_l1 = 0.0
    total_base_abs = 0.0
    n_batches = 0

    for batch in loader:
        x = batch["input"].to(device, non_blocking=True)
        y = batch["label"].to(device, non_blocking=True)
        b = batch["baseline"].to(device, non_blocking=True)

        pred = model(x, b)

        loss, loss_l1, loss_abs = compute_loss(pred, y)

        base_l1 = F.l1_loss(b, y)
        base_abs = F.l1_loss(complex_abs_2ch(b), complex_abs_2ch(y))

        total_loss += loss.item()
        total_l1 += loss_l1.item()
        total_abs += loss_abs.item()
        total_base_l1 += base_l1.item()
        total_base_abs += base_abs.item()
        n_batches += 1

    total_loss /= max(n_batches, 1)
    total_l1 /= max(n_batches, 1)
    total_abs /= max(n_batches, 1)
    total_base_l1 /= max(n_batches, 1)
    total_base_abs /= max(n_batches, 1)

    improvement = 1.0 - total_l1 / (total_base_l1 + 1e-12)

    return {
        "loss": total_loss,
        "l1": total_l1,
        "abs": total_abs,
        "baseline_l1": total_base_l1,
        "baseline_abs": total_base_abs,
        "improvement": improvement,
    }


def train_model_jupyter(
    model,
    train_loader,
    val_loader,
    device,
    save_dir="checkpoints_grouped_rfnet",
    num_epochs=100,
    lr=1e-3,
    weight_decay=1e-5,
    print_every=5,
):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_epochs,
        eta_min=1e-5,
    )

    best_val_l1 = float("inf")
    best_epoch = -1
    history = []

    init_val = evaluate_normalized(model, val_loader, device)
    print("\nInitial validation:")
    print(
        f"val_L1={init_val['l1']:.6e} | "
        f"baseline_L1={init_val['baseline_l1']:.6e} | "
        f"improvement={init_val['improvement']*100:.2f}%"
    )

    for epoch in range(1, num_epochs + 1):
        model.train()

        train_l1 = 0.0
        train_abs = 0.0
        train_loss = 0.0
        train_base = 0.0
        n_batches = 0

        for batch in train_loader:
            x = batch["input"].to(device, non_blocking=True)
            y = batch["label"].to(device, non_blocking=True)
            b = batch["baseline"].to(device, non_blocking=True)

            pred = model(x, b)
            loss, loss_l1, loss_abs = compute_loss(pred, y)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            with torch.no_grad():
                base_l1 = F.l1_loss(b, y)

            train_loss += loss.item()
            train_l1 += loss_l1.item()
            train_abs += loss_abs.item()
            train_base += base_l1.item()
            n_batches += 1

        scheduler.step()

        train_loss /= max(n_batches, 1)
        train_l1 /= max(n_batches, 1)
        train_abs /= max(n_batches, 1)
        train_base /= max(n_batches, 1)

        train_impr = 1.0 - train_l1 / (train_base + 1e-12)

        val_metrics = evaluate_normalized(model, val_loader, device)

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_l1": train_l1,
            "train_abs": train_abs,
            "train_base": train_base,
            "train_impr": train_impr,
            "val_l1": val_metrics["l1"],
            "val_abs": val_metrics["abs"],
            "val_base": val_metrics["baseline_l1"],
            "val_impr": val_metrics["improvement"],
            "lr": scheduler.get_last_lr()[0],
        }
        history.append(row)

        if val_metrics["l1"] < best_val_l1:
            best_val_l1 = val_metrics["l1"]
            best_epoch = epoch

            torch.save(
                {
                    "epoch": epoch,
                    "model": model.state_dict(),
                    "best_val_l1": best_val_l1,
                    "history": history,
                },
                save_dir / "best_grouped_rf_residual_net.pth",
            )

        if epoch == 1 or epoch % print_every == 0:
            print(
                f"Epoch {epoch:04d} | "
                f"train_L1={train_l1:.6e} | "
                f"train_base={train_base:.6e} | "
                f"train_impr={train_impr*100:6.2f}% | "
                f"val_L1={val_metrics['l1']:.6e} | "
                f"val_base={val_metrics['baseline_l1']:.6e} | "
                f"val_impr={val_metrics['improvement']*100:6.2f}% | "
                f"lr={scheduler.get_last_lr()[0]:.2e}"
            )

    torch.save(
        model.state_dict(),
        save_dir / "final_grouped_rf_residual_net.pth",
    )

    print("\nTraining finished.")
    print(f"Best val L1 = {best_val_l1:.6e} at epoch {best_epoch}")
    print(f"Best checkpoint: {save_dir / 'best_grouped_rf_residual_net.pth'}")

    history_df = pd.DataFrame(history)
    return history_df

In [ ]:
seed_everything(20260522)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

earlymix_model = EarlyMixWideResidualNet(
    in_channels=1536,
    stem_channels=128,
    mid_channels=64,
    out_channels=2,
).to(device)

n_params = sum(p.numel() for p in earlymix_model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params/1e6:.3f} M")

history_earlymix = train_model_jupyter(
    model=earlymix_model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    save_dir="checkpoints_earlymix_wide",
    num_epochs=100,
    lr=1e-3,
    weight_decay=1e-5,
    print_every=5,
)